# Module 4 — Feature Engineering
ML Engineer (modelling) role.

1. Aggregate the **cleaned** synthetic logs and integrate them with OULAD (one row per enrollment)
2. Build **RFM-style** behavioural features: Recency of last activity, Frequency of activity, engagement
   intensity ("Monetary"), plus consistency, pacing, performance, content, social and rhythm features
3. Encode categorical variables (nominal → binary, ordinal → codes; Lecture 7)
4. Transform skewed features and compare scalers (Min-Max vs Z-score vs robust; Lecture 7)
5. Select the feature set (variance, redundancy filter, interpretability) and check PCA (dimensionality reduction)
6. Fit the transformation pipeline **on training cohorts only** and save it for serving

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src")) if (Path.cwd().parent / "src").exists() else sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, PowerTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from edtech import config as C, features as F, utils as U, viz as V
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)

enr = pd.read_parquet(C.INTERIM / "oulad_enrollments_clean.parquet")
sess = pd.read_parquet(C.INTERIM / "sessions_clean.parquet")
subs = pd.read_parquet(C.INTERIM / "submissions_clean.parquet")

## 1. Integration of cleaned sources

In [2]:
df = F.integrate(enr, F.session_aggregates(sess, enr), F.submission_aggregates(subs))
assert len(df) == len(enr) and df.duplicated(C.KEY).sum() == 0
df["split"] = np.where(df.code_presentation.isin(C.TRAIN_PRESENTATIONS), "train", "monitor")
df["population"] = np.where(df.no_activity == 1, "non-starter (rule-based)", "clustered")
print(df.shape)
pd.crosstab(df.population, df.split, margins=True)

(32593, 97)


split,monitor,train,All
population,,,
clustered,9830,18670,28500
non-starter (rule-based),1430,2663,4093
All,11260,21333,32593


**Population decision.** 4,093 enrollments (12.6%) have *no* VLE activity in the course window. They have no
recency, no pacing and no intensity — any value we put into a distance calculation would be invented, and they
would form a trivial cluster that distorts K-Means centroids (Lecture 9: K-Means is sensitive to such masses).
They are therefore assigned by a transparent **business rule** to segment **S0 "Non-starters"**, and clustering is
run on the 28,500 learners with activity. **Temporal split:** presentations 2013B–2014B train the model; 2014J
(11,260 enrollments) is held out as the "incoming cohort" for validation and monitoring.

## 2. Candidate features (RFM-style and beyond)

In [3]:
cand = pd.DataFrame({"feature": list(F.CANDIDATES), "role": list(F.CANDIDATES.values())})
cand["source"] = np.where(cand.feature.isin(["logins_per_week", "login_week_coverage", "last_minute_submission_share",
                                              "median_hours_before_deadline", "video_completion_rate", "forum_posts_per_week",
                                              "night_login_share", "days_to_first_login", "mobile_share"]), "synthetic", "OULAD")
act = df[df.population == "clustered"]
cand["missing_pct"] = cand.feature.map(lambda c: round(100 * act[c].isna().mean(), 2))
cand["coef_variation"] = cand.feature.map(lambda c: round(act[c].std() / abs(act[c].mean()), 2))
cand["skew"] = cand.feature.map(lambda c: round(act[c].skew(), 2))
U.save_table(cand, "m4_candidate_features"); cand

,feature,role,source,missing_pct,coef_variation,skew
0,recency_fraction,R - share of course elapsed since last activity,OULAD,0.00,1.16,0.91
1,active_day_fraction,F - share of course days with VLE activity,OULAD,0.00,0.87,1.09
2,logins_per_week,F - synthetic login frequency,synthetic,0.00,0.91,1.48
3,login_week_coverage,F - share of weeks with at least one login,synthetic,0.00,0.56,-0.22
4,total_clicks,M - recorded resource interactions,OULAD,0.00,1.27,2.93
5,clicks_per_week,M - interaction intensity,OULAD,0.00,1.27,2.91
6,clicks_per_active_day,M - depth per study day,OULAD,0.00,0.61,1.99
7,weekly_click_cv,Consistency - variability of weekly clicks (whole course),OULAD,0.00,0.62,1.33
8,weekly_cv_active_span,Consistency - variability of weekly clicks within own active span,OULAD,0.00,0.48,0.92
9,late_click_share,Pacing - share of clicks in last third,OULAD,0.00,1.06,1.09


## 3. Encoding categorical variables
* **Nominal → binary (1-to-N)** for module, domain, start month, gender, disability, age band and education
  (Lecture 7: integer codes would invent an order and distances between categories).
* **Ordinal → numeric** for IMD decile, education level and age band where order is meaningful.

Demographics are **not** clustering inputs: segments must describe behaviour that the platform can influence, and
using protected attributes in a targeting model raises fairness concerns. Encoded context is used to *profile*
segments (Module 5) and to check for demographic skew.

In [4]:
onehot = F.one_hot_context(df)
ordinal = F.ordinal_context(df)
onehot.to_parquet(C.PROCESSED / "context_onehot.parquet", index=False)
ordinal.to_parquet(C.PROCESSED / "context_ordinal.parquet", index=False)
print("one-hot columns:", onehot.shape[1] - 3); ordinal.describe().round(2)

one-hot columns: 23


,id_student,imd_decile,education_level,age_level
count,32593.00,31482.00,32593.00,32593.00
mean,706687.67,5.21,1.74,0.30
std,549167.31,2.82,0.75,0.47
min,3733.00,1.00,0.00,0.00
25%,508573.00,3.00,1.00,0.00
50%,590310.00,5.00,2.00,0.00
75%,644453.00,8.00,2.00,1.00
max,2716795.00,10.00,4.00,2.00


## 4. Transformation and scaling
### 4a. Spreading skewed distributions (Lecture 7: log, Box-Cox, rank transforms)

In [5]:
skewed = ["total_clicks", "clicks_per_active_day", "weekly_cv_active_span", "deadline_click_ratio", "forum_posts_per_week", "logins_per_week"]
rows = []
for c in skewed:
    x = act[c].astype(float).dropna()
    r = {"feature": c, "raw": stats.skew(x), "log1p": stats.skew(np.log1p(x))}
    r["box_cox(x+1)"] = stats.skew(stats.boxcox(x + 1)[0])
    r["yeo_johnson"] = stats.skew(PowerTransformer(standardize=False).fit_transform(x.to_frame()).ravel())
    r["rank_normal"] = stats.skew(stats.norm.ppf((x.rank() - 0.5) / len(x)))
    rows.append(r)
sk = pd.DataFrame(rows).round(3); U.save_table(sk, "m4_skewness_by_transform"); sk

,feature,raw,log1p,box_cox(x+1),yeo_johnson,rank_normal
0,total_clicks,2.927,-0.926,-0.036,-0.036,0.012
1,clicks_per_active_day,1.994,-0.366,0.017,0.017,0.013
2,weekly_cv_active_span,0.919,-0.446,0.042,0.042,0.076
3,deadline_click_ratio,1.839,0.108,0.007,0.007,0.180
4,forum_posts_per_week,9.471,3.594,0.827,0.827,0.649
5,logins_per_week,1.481,0.191,0.031,0.031,0.013


In [6]:
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for i, c in enumerate(["clicks_per_active_day", "deadline_click_ratio", "forum_posts_per_week"]):
    x = act[c].astype(float)
    ax[0, i].hist(x.clip(upper=x.quantile(.995)), bins=60, color=V.CATEGORICAL[7]); ax[0, i].set_title(f"{c} (raw)", fontsize=10)
    ax[1, i].hist(PowerTransformer(standardize=False).fit_transform(x.to_frame()).ravel(), bins=60, color=V.CATEGORICAL[0])
    ax[1, i].set_title(f"{c} (Yeo-Johnson)", fontsize=10)
fig.tight_layout(); V.save(fig, "m4_transformations"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15772\1756920456.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); V.save(fig, "m4_transformations"); plt.show()


Yeo-Johnson (the Box-Cox family generalised to zero values) brings skewness closest to 0 for all features and
handles the many zeros in forum posts; λ is estimated on training data only. Rank-normal scores are excellent
too but, as Lecture 7 notes, cannot be applied separately to training and new data, so they are unsuitable for a
deployed model.

### 4b. Scaling: why Z-score

In [7]:
sel = F.SELECTED
Xr = act[sel].astype(float).fillna(act[sel].median())
Xt = Xr.copy(); sk_sel = [c for c in sel if c in F.SKEWED]
Xt[sk_sel] = PowerTransformer(standardize=False).fit_transform(Xt[sk_sel])
share = {}
for name, sc in [("none", None), ("min-max", MinMaxScaler()), ("z-score", StandardScaler()), ("robust (IQR)", RobustScaler())]:
    Z = Xt.to_numpy() if sc is None else sc.fit_transform(Xt)
    v = Z.var(0); share[name] = v / v.sum()
contrib = pd.DataFrame(share, index=sel).round(3)
U.save_table(contrib.reset_index(names="feature"), "m4_variance_share_by_scaler"); contrib

,none,min-max,z-score,robust (IQR)
recency_fraction,0.000,0.169,0.1,0.075
active_day_fraction,0.000,0.062,0.1,0.108
clicks_per_active_day,0.001,0.019,0.1,0.116
weekly_cv_active_span,0.000,0.028,0.1,0.156
deadline_click_ratio,0.000,0.065,0.1,0.120
avg_score_scheduled,0.998,0.156,0.1,0.056
last_minute_submission_share,0.000,0.106,0.1,0.158
video_completion_rate,0.000,0.064,0.1,0.089
forum_posts_per_week,0.000,0.175,0.1,0.075
night_login_share,0.000,0.156,0.1,0.047


Without scaling, `avg_score_scheduled` (0–100 points) carries more than 99% of the variance and would dominate
Euclidean distance (Lecture 7: normalisation prevents large-range attributes from outweighing
small-range ones). Min-Max still leaves unequal weights because of outliers (it "does not eliminate outliers");
**Z-score after Yeo-Johnson gives every feature equal weight (10%)**, which is the intended neutral starting point.

## 5. Feature selection
### 5a. Redundancy filter (|Spearman ρ| ≥ 0.80, Lecture 9: "minimal information redundancy")

In [8]:
red = F.correlation_filter(act, list(F.CANDIDATES), 0.80)
U.save_table(red, "m4_redundant_pairs"); red

,feature_a,feature_b,spearman_rho
0,recency_fraction,login_week_coverage,-0.870
1,active_day_fraction,logins_per_week,0.985
2,active_day_fraction,login_week_coverage,0.961
3,active_day_fraction,total_clicks,0.917
4,active_day_fraction,clicks_per_week,0.919
5,active_day_fraction,weekly_click_cv,-0.911
6,logins_per_week,login_week_coverage,0.946
7,logins_per_week,total_clicks,0.916
8,logins_per_week,clicks_per_week,0.918
9,logins_per_week,weekly_click_cv,-0.895


In [9]:
decisions = pd.DataFrame([
    ("recency_fraction", "keep", "R of RFM; ρ = −0.80 with active days is at the threshold but measures a distinct concept (drop-off timing)"),
    ("active_day_fraction", "keep", "F of RFM; real data; represents the frequency block"),
    ("logins_per_week / login_week_coverage", "drop", "ρ ≥ 0.96 with active_day_fraction — same information (synthetic anchored to it)"),
    ("total_clicks / clicks_per_week", "drop", "ρ ≥ 0.92 with active_day_fraction"),
    ("clicks_per_active_day", "keep", "M of RFM: depth of each study day, ρ ≈ 0.3 with frequency"),
    ("weekly_click_cv", "replace", "ρ = −0.91 with frequency (low activity ⇒ high CV); replaced by weekly_cv_active_span"),
    ("weekly_cv_active_span", "keep", "burstiness inside the learner's own active period — independent of volume"),
    ("deadline_click_ratio", "keep", "pacing: concentration of study on deadline days (real)"),
    ("late_click_share", "drop", "mostly measures persistence (ρ 0.75 with submission), not cramming"),
    ("avg_score_scheduled", "keep", "performance; combines submission and marks"),
    ("submission_rate / on_time_rate_scheduled", "drop", "ρ 0.79–0.90 with avg_score_scheduled"),
    ("last_minute_submission_share", "keep", "timeliness at hour level (synthetic) — the crammer signal"),
    ("median_hours_before_deadline", "drop", "missing for non-submitters; overlaps last_minute share"),
    ("video_completion_rate", "keep", "content engagement (synthetic)"),
    ("forum_posts_per_week", "keep", "social engagement (synthetic)"),
    ("night_login_share", "keep", "study rhythm (synthetic); independent of all others"),
    ("days_to_first_login / mobile_share", "profile only", "onboarding & context — useful to describe, not to define, behaviour"),
], columns=["feature", "decision", "reason"])
U.save_table(decisions, "m4_selection_decisions"); decisions

,feature,decision,reason
0,recency_fraction,keep,R of RFM; ρ = −0.80 with active days is at the threshold but measures a dist...
1,active_day_fraction,keep,F of RFM; real data; represents the frequency block
2,logins_per_week / login_week_coverage,drop,ρ ≥ 0.96 with active_day_fraction — same information (synthetic anchored to it)
3,total_clicks / clicks_per_week,drop,ρ ≥ 0.92 with active_day_fraction
4,clicks_per_active_day,keep,"M of RFM: depth of each study day, ρ ≈ 0.3 with frequency"
5,weekly_click_cv,replace,ρ = −0.91 with frequency (low activity ⇒ high CV); replaced by weekly_cv_act...
6,weekly_cv_active_span,keep,burstiness inside the learner's own active period — independent of volume
7,deadline_click_ratio,keep,pacing: concentration of study on deadline days (real)
8,late_click_share,drop,"mostly measures persistence (ρ 0.75 with submission), not cramming"
9,avg_score_scheduled,keep,performance; combines submission and marks


In [10]:
corr_sel = act[sel].corr("spearman")
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr_sel, cmap=V.DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(len(sel)), sel, rotation=60, ha="right"); ax.set_yticks(range(len(sel)), sel)
for i in range(len(sel)):
    for j in range(len(sel)):
        ax.text(j, i, f"{corr_sel.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.grid(False); fig.colorbar(im, shrink=0.7); ax.set_title("Selected features — Spearman ρ")
V.save(fig, "m4_selected_correlation"); plt.show()
print("max |rho| off-diagonal:", (corr_sel.abs() - np.eye(len(sel))).max().max().round(3))

max |rho| off-diagonal: 0.799


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15772\3456075902.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m4_selected_correlation"); plt.show()


### 5b. Fit the pipeline on training cohorts only

In [11]:
train = df[(df.split == "train") & (df.population == "clustered")]
pipe = F.FeaturePipeline(sel).fit(train)
Z_train = pipe.transform(train)
print("train matrix:", Z_train.shape, "| mean≈0:", np.abs(Z_train.mean(0)).max().round(6), "| sd≈1:", Z_train.std(0).round(3))
joblib.dump(pipe, C.MODELS / "feature_pipeline.joblib")
params = pd.DataFrame({"feature": sel, "median_fill": [pipe.medians_[c] for c in sel],
                       "clip_low_p0.5": [pipe.clip_[c][0] for c in sel], "clip_high_p99.5": [pipe.clip_[c][1] for c in sel],
                       "yeo_johnson_lambda": [dict(zip(pipe.skewed, pipe.power.lambdas_)).get(c, np.nan) for c in sel],
                       "z_mean": pipe.scaler.mean_, "z_scale": pipe.scaler.scale_}).round(4)
U.save_table(params, "m4_pipeline_parameters"); params

train matrix: (18670, 10) | mean≈0: 0.0 | sd≈1: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


,feature,median_fill,clip_low_p0.5,clip_high_p99.5,yeo_johnson_lambda,z_mean,z_scale
0,recency_fraction,0.0929,0.0000,0.9885,NaN,0.2846,0.3317
1,active_day_fraction,0.1822,0.0037,0.9107,NaN,0.2373,0.2042
2,clicks_per_active_day,16.1180,1.5000,68.5818,0.1396,3.5390,0.8300
3,weekly_cv_active_span,1.1833,0.0000,3.5885,0.3899,0.9256,0.3589
4,deadline_click_ratio,1.8865,0.0000,12.3511,-0.0390,1.0439,0.5856
5,avg_score_scheduled,61.0909,0.0000,96.1041,NaN,50.7586,32.2635
6,last_minute_submission_share,0.0000,0.0000,1.0000,NaN,0.1815,0.2602
7,video_completion_rate,0.6393,0.1410,0.9620,NaN,0.6194,0.1885
8,forum_posts_per_week,0.0294,0.0000,4.7863,-4.7675,0.0647,0.0711
9,night_login_share,0.1623,0.0000,1.0000,NaN,0.2892,0.3211


### 5c. Dimensionality reduction check (PCA)

In [12]:
pca = PCA().fit(Z_train)
ev = pd.DataFrame({"component": [f"PC{i+1}" for i in range(len(sel))], "explained_variance_ratio": pca.explained_variance_ratio_,
                   "cumulative": pca.explained_variance_ratio_.cumsum()}).round(3)
load = pd.DataFrame(pca.components_[:4].T, index=sel, columns=["PC1", "PC2", "PC3", "PC4"]).round(2)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(ev.component, ev.explained_variance_ratio, color=V.CATEGORICAL[0], width=0.6)
ax[0].plot(ev.component, ev.cumulative, color=V.CATEGORICAL[1], marker="o", label="cumulative")
ax[0].axhline(0.8, color=V.TEXT2, ls=":", lw=1); ax[0].set(title="PCA scree plot (training cohorts)", ylabel="explained variance"); ax[0].legend()
idx = np.random.default_rng(0).choice(len(Z_train), 6000, replace=False)
P = pca.transform(Z_train)[idx]
sc = ax[1].scatter(P[:, 0], P[:, 1], c=train.avg_score_scheduled.to_numpy()[idx], cmap=V.SEQUENTIAL, s=5, alpha=0.6)
ax[1].set(title="PC1 vs PC2 coloured by average score", xlabel="PC1", ylabel="PC2"); fig.colorbar(sc, ax=ax[1], label="avg score")
V.save(fig, "m4_pca"); plt.show()
U.save_table(ev, "m4_pca_explained"); U.save_table(load.reset_index(names="feature"), "m4_pca_loadings")
load

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15772\3756919465.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m4_pca"); plt.show()


,PC1,PC2,PC3,PC4
recency_fraction,-0.46,-0.25,0.13,0.27
active_day_fraction,0.49,-0.13,-0.02,0.17
clicks_per_active_day,0.22,0.05,0.77,0.14
weekly_cv_active_span,-0.01,0.55,0.14,-0.50
deadline_click_ratio,0.02,0.57,0.03,0.21
avg_score_scheduled,0.48,0.13,-0.13,-0.20
last_minute_submission_share,-0.00,0.40,-0.13,0.66
video_completion_rate,0.32,-0.22,-0.38,-0.01
forum_posts_per_week,0.41,-0.13,0.14,0.28
night_login_share,-0.05,0.24,-0.42,0.15


In [13]:
# Does clustering in a reduced PCA space help? Compare K-Means (k=4) on the full matrix vs on PCs reaching 80% variance.
n80 = int((pca.explained_variance_ratio_.cumsum() < 0.80).sum() + 1)
res = []
for name, M in [("all 10 standardised features", Z_train), (f"first {n80} PCs (80% var)", pca.transform(Z_train)[:, :n80])]:
    lab = KMeans(4, n_init=10, random_state=C.SEED).fit_predict(M)
    res.append({"space": name, "dims": M.shape[1], "silhouette_in_own_space": silhouette_score(M[idx], lab[idx]),
                "silhouette_in_original_space": silhouette_score(Z_train[idx], lab[idx])})
pca_cmp = pd.DataFrame(res).round(3); U.save_table(pca_cmp, "m4_pca_vs_full"); pca_cmp

,space,dims,silhouette_in_own_space,silhouette_in_original_space
0,all 10 standardised features,10,0.171,0.171
1,first 5 PCs (80% var),5,0.232,0.179


PC1 is a general "engagement & performance" axis; PC2 is a "pacing" axis (deadline concentration, last-minute
submissions, burstiness, night logins) — confirming the two behavioural dimensions found in EDA. Ten features are
few, there is no severe multicollinearity left after selection, and reaching 80% of the variance still needs about
half of the components (PC1 33%, PC2 19%). Clustering on the first 5 PCs looks better *in its own space*
(silhouette 0.23 — dropped components shrink distances) but, judged in the original feature space, the partition
is practically the same quality (0.179 vs 0.171) while centroids would lose their direct meaning. **Decision: cluster on the 10 standardised features; use PCA for
visualisation** — this also keeps centroids directly interpretable for personas.

## 6. Save the feature store

In [14]:
Z_all = pipe.transform(df)
mat = pd.DataFrame(Z_all, columns=[f"z_{c}" for c in sel])
feat = pd.concat([df.reset_index(drop=True), mat], axis=1)
feat.to_parquet(C.PROCESSED / "learner_features.parquet", index=False)
print(feat.shape)

fig, ax = plt.subplots(figsize=(8, 4))
for o in ["Distinction", "Pass", "Fail", "Withdrawn"]:
    ax.hist(df.loc[df.final_result == o, "engagement_score"], bins=50, histtype="step", lw=2, color=V.OUTCOME_COLORS[o], label=o, density=True)
ax.set(title="Engagement score KPI by final result", xlabel="engagement score (0-100)", ylabel="density"); ax.legend()
V.save(fig, "m4_engagement_score"); plt.show()

U.save_metrics("m4", {"selected_features": sel, "n_clustered": int((df.population == "clustered").sum()),
                      "n_non_starters": int((df.population != "clustered").sum()),
                      "n_train_clustered": len(train), "n_monitor": int((df.split == "monitor").sum()),
                      "pca_explained": ev.explained_variance_ratio.tolist(), "pcs_for_80pct": n80,
                      "max_abs_rho_selected": float((corr_sel.abs() - np.eye(len(sel))).max().max()),
                      "engagement_score_by_outcome": df.groupby("final_result").engagement_score.median().round(1).to_dict(),
                      "pca_vs_full": pca_cmp.to_dict("records")})

(32593, 107)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15772\2210843525.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m4_engagement_score"); plt.show()
